# AgentMorph - Stage 3 FULL SWEEP (overnight)

The 1,000-pair production run. Gate: the Apr 27 dress rehearsal (`stage3_dress_rehearsal.ipynb`) must be green before launching this.

**Scope (per 14-day plan):**
- 5 models: Llama-3.2-3B, Qwen2.5-7B, Gemma-2-9B, Phi-4, Llama-3.1-8B
- 10 rules: all of `RULE_IDS`
- 2 frameworks: native + langgraph (smolagents excluded; blows T4 context by step 3)
- 1 env: ecommerce
- 20 scenarios (with per-rule filter; refusal-consistency sees only 2, most others see 18-20)

**Expected cell count** (after filtering, matches 14-day plan):
- refusal-consistency: 5 models x 2 frameworks x 2 scenarios = 20 cells (3-way = 60 trajectories)
- other 9 rules:       5 models x 2 frameworks x 18 scenarios = 1,620 cells
- **Total: ~1,640 cells, ~3,260 trajectories**

**Wall-clock on T4:** ~17 hours. **Kick at 18:00**, runs into next day noon.

**Colab budget note:** one overnight run comfortably fits in one Colab session. If it kills mid-run, the manifest means re-launching this notebook continues exactly where it stopped.

**Before you run:** Runtime > Change runtime type > **T4 High-RAM GPU** (the 9B Gemma + Phi-4 both need the High-RAM tier). Have HF token ready for Section 3.

## 1. GPU sanity check

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit('CUDA required for the full sweep - switch to T4 High-RAM and restart.')
dev = torch.cuda.get_device_properties(0)
print(f'CUDA OK: {torch.cuda.get_device_name(0)} ({dev.total_memory / 1024**3:.1f} GB VRAM)')
if dev.total_memory / 1024**3 < 14.5:
    print('WARN: VRAM < 14.5 GB - Phi-4 / Gemma-2-9B may OOM. Use T4 High-RAM.')

## 2. Clone / pull the repo

In [ ]:
import os
REPO_URL = "https://github.com/Pranamya16/AgentMorph.git"
REPO_DIR = "/content/AgentMorph"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout main && git pull --ff-only
!cd {REPO_DIR} && git log -1 --oneline

## 3. HuggingFace auth

Paste your token, run, clear the token.

In [ ]:
from huggingface_hub import login
HF_TOKEN = "hf_REPLACE_ME"
login(token=HF_TOKEN)

## 4. Mount Drive + install all extras

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q -e /content/AgentMorph[models,smolagents,langgraph,gemini]

## 5. Verify the paraphrase cache

Hard dependency: rules 2, 3, 5 all raise `ParaphraseCacheMiss` in offline mode without this. Expect 30 + 20 + 4 = 54 entries across 3 files.

In [ ]:
import pathlib
cache_dir = pathlib.Path('/content/AgentMorph/runs/paraphrase_cache')
if not cache_dir.exists():
    raise SystemExit('paraphrase cache missing - re-run stage2_seed_paraphrases.ipynb')
files = sorted(cache_dir.glob('*.jsonl'))
assert len(files) == 3, f'expected 3 cache files, got {len(files)}'
counts = {p.name: sum(1 for _ in p.open()) for p in files}
for k, v in counts.items():
    print(f'{k}: {v} entries')
assert counts.get('schema_paraphrase_invariance.jsonl', 0) >= 30, 'schema cache incomplete'
assert counts.get('synonym_robustness.jsonl', 0) >= 20, 'synonym cache incomplete'
assert counts.get('refusal_consistency.jsonl', 0) >= 4, 'refusal cache incomplete'
print('cache OK')

## 6. Sweep parameters (full production scope)

In [ ]:
OUT_DIR = "/content/drive/MyDrive/AgentMorph/runs/stage3_baseline"
HF_CACHE = "/content/drive/MyDrive/AgentMorph/hf_cache"
MODELS = ["Llama-3.2-3B", "Qwen2.5-7B", "Gemma-2-9B", "Phi-4", "Llama-3.1-8B"]
FRAMEWORKS = ["native", "langgraph"]
# Leaving RULES unset -> runner uses all 10 from RULE_IDS.
N_SCENARIOS = 20
print('full sweep config:')
print('  models    :', MODELS)
print('  frameworks:', FRAMEWORKS)
print('  rules     : all 10 (RULE_IDS)')

## 7. Dry-run: verify the cell count matches the plan

Target: ~1,640 cells (after per-rule filtering). Manifest replays already-completed cells as skipped.

In [ ]:
def _flags():
    f = []
    for m in MODELS: f += ['--model', m]
    for fw in FRAMEWORKS: f += ['--framework', fw]
    return f
flags = ' '.join(_flags())
!cd /content/AgentMorph && python -m agentmorph.runner --stage3 --dry-run \
    {flags} --env ecommerce --n-scenarios {N_SCENARIOS} \
    --out-dir {OUT_DIR}

## 8. Kick the overnight run

Streams progress to stdout. If the Colab runtime dies, re-run this cell from a fresh runtime - the manifest picks up from the last completed cell. One cell loss per kill, max.

**Leave this running.** Do not close the Colab tab.

Per-rule timing estimate:
- non-refusal rule cell: ~60 s (2 trajectories of up to 3 steps each)
- refusal rule cell    : ~90 s (3 trajectories)
- 1,620 * 60s + 20 * 90s = 28.3 h total without KV reuse; actual ~17 h with VRAM reclaim.


In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
flags = ' '.join(_flags())
!cd /content/AgentMorph && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
    python -m agentmorph.runner --stage3 \
    {flags} --env ecommerce --n-scenarios {N_SCENARIOS} \
    --capture-state \
    --hf-cache-dir {HF_CACHE} \
    --out-dir {OUT_DIR} 2>&1 | tee /content/drive/MyDrive/AgentMorph/runs/stage3_baseline/sweep.log

## 9. Post-run tally

Call this after Section 8 finishes OR after a kill+resume cycle to see current progress.

In [ ]:
import json, collections, pathlib
out = pathlib.Path(OUT_DIR)
manifest = json.loads((out / 'manifest.json').read_text()) if (out / 'manifest.json').exists() else {'completed': {}}
completed = manifest.get('completed', {})
print(f'cells completed: {len(completed)}')

bug_manifest_rows = sum(1 for v in completed.values() if v.get('is_bug'))
print(f'bug cells (from manifest): {bug_manifest_rows}')

bugs_path = out / 'bugs.jsonl'
if bugs_path.exists():
    bugs = [json.loads(l) for l in bugs_path.open()]
    print(f'bugs.jsonl rows: {len(bugs)}')
    by_rule = collections.Counter(b['rule_id'] for b in bugs)
    by_dv = collections.Counter(b['divergence_type'] for b in bugs)
    by_model = collections.Counter(b['model_id'] for b in bugs)
    print('\nby rule:')
    for k, v in sorted(by_rule.items()):
        print(f'  {k:38s} {v}')
    print('\nby divergence type:')
    for k, v in sorted(by_dv.items()):
        print(f'  {k:25s} {v}')
    print('\nby model:')
    for k, v in sorted(by_model.items()):
        print(f'  {k:20s} {v}')

# Pair files summary.
traj_dir = out / 'trajectories'
if traj_dir.exists():
    files = sorted(traj_dir.glob('*.jsonl'))
    total_pairs = sum(sum(1 for _ in p.open()) for p in files)
    print(f'\nper-cell trajectory JSONL files: {len(files)}')
    print(f'total pair records: {total_pairs}')

## 10. Post-sweep integrity checks

Run before handing the data off to the Stage 4 classifier / HF uploader.

In [ ]:
import json, pathlib
out = pathlib.Path(OUT_DIR)
traj_dir = out / 'trajectories'

# 1) Every pair line parses.
bad_lines = 0
for p in traj_dir.glob('*.jsonl'):
    with p.open() as fh:
        for i, line in enumerate(fh, 1):
            try: json.loads(line)
            except Exception: bad_lines += 1
print(f'pair JSONL parse errors: {bad_lines}')

# 2) Every bug row round-trips through Bug.from_dict.
from agentmorph.rules.base import Bug
bugs_path = out / 'bugs.jsonl'
bad_bugs = 0
if bugs_path.exists():
    with bugs_path.open() as fh:
        for line in fh:
            try:
                Bug.from_dict(json.loads(line))
            except Exception as exc:
                bad_bugs += 1
                print('bad bug row:', exc)
print(f'bug parse errors: {bad_bugs}')

# 3) bug_id uniqueness (each pair id should appear <=1 time in bugs.jsonl)
if bugs_path.exists():
    ids = [json.loads(l)['bug_id'] for l in bugs_path.open()]
    dup = len(ids) - len(set(ids))
    print(f'duplicate bug_ids: {dup}')

## 11. Handoff

At this point `runs/stage3_baseline/` on Drive contains:
- `manifest.json` (one row per completed cell)
- `trajectories/*.jsonl` (one file per (model, framework, env, rule))
- `bugs.jsonl` (one row per divergent pair - the HF dataset seed)
- `sweep.log` (full stdout/stderr)

**Next steps (owned by Claude Code / user split per `CLAUDE.md`):**
1. `scripts/classify_bugs.py` - stratified 50-bug sample for chat-side severity labelling.
2. `scripts/upload_to_hf.py` - push `bugs.jsonl` to `agentmorph/bugs-v0.1` on HuggingFace.
3. `scripts/render_figures.py` - the 4 paper figures (bug heatmap, finish-rate heatmap, transfer bar, rule-x-model heatmap).
4. Transfer study via `src/agentmorph/transfer.py` (Gemini 2.5 Flash, Claude Haiku, Groq Llama-3.3-70B) on 100 sampled bugs.
